In [1]:

import os
import re
import sys
import gc
import random
import subprocess
from collections import defaultdict

# ---------------------------------------------------------
# 0. Environment setup (safe, minimal, no forced version pins)
# ---------------------------------------------------------
def setup_environment():
    """
    Installs only the packages that are typically missing on a fresh
    Kaggle GPU notebook. Does NOT force-reinstall transformers/tokenizers/
    torch/pandas - Kaggle's base image already ships a torch build that
    matches the installed CUDA driver, and force-reinstalling unrelated
    pinned versions risks breaking that match.
    """
    packages = [
        "datasets>=2.19.0",
        "evaluate>=0.4.2",
        "rouge_score>=0.1.2",
        "accelerate>=0.33.0",
        "sentencepiece>=0.2.0",
    ]
    print("[Setup] Installing helper packages (no force-reinstall, no version pins on core libs)...")
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


setup_environment()

import torch
import numpy as np
import pandas as pd
import evaluate
import transformers
import tokenizers
from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    set_seed,
)

# ---------------------------------------------------------
# 1. Reproducibility & hardware setup
# ---------------------------------------------------------
SEED = 42


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    # NOTE: cudnn.benchmark is intentionally NOT enabled here.
    # It picks conv algorithms non-deterministically, which contradicts
    # the fixed-seed reproducibility this function is meant to provide.
    # T5 also has no conv layers, so it would buy no speed anyway.


seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\n=== System Infrastructure ===")
print(f"Device Target: {device}")
if device.type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Hardware: {gpu_name}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"Transformers Version: {transformers.__version__}")
    print(f"Tokenizers Version: {tokenizers.__version__}")
    torch.cuda.empty_cache()
else:
    print("[WARNING] CUDA is not available. Running on CPU will be slow.")

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
CHECKPOINT_DIR = os.path.join(WORK_DIR, "t5_qg_checkpoint")
SAVE_DIR = os.path.join(WORK_DIR, "t5_qg_finetuned_final")

# ---------------------------------------------------------
# 2. Text processing / prompt engineering
# ---------------------------------------------------------
def highlight_answer_robust(context: str, answer: str) -> str:
    """
    Highlights the answer span using <hl> answer <hl> format, matching
    the format valhalla/t5-small-qg-hl was trained on.

    Uses a replacement FUNCTION (not a raw string) so that backslashes
    or other regex-special characters inside `answer` are treated as
    literal text instead of being interpreted by re.subn as
    backreferences (e.g. \\1, \\g<0>), which could otherwise raise
    re.error or silently corrupt the output.
    """
    if not answer or not context:
        return context

    pattern = re.escape(answer)
    highlighted, count = re.subn(
        pattern,
        lambda m: f" <hl> {answer} <hl> ",
        context,
        count=1,
        flags=re.IGNORECASE,
    )

    if count == 0:
        highlighted = f" <hl> {answer} <hl> {context}"

    return " ".join(highlighted.split())


# ---------------------------------------------------------
# 3. Dataset loading
# ---------------------------------------------------------
print("\n=== Dataset Loading (SQuAD v1.1) ===")
try:
    raw_train = load_dataset("rajpurkar/squad", split="train[:5000]")
    raw_eval = load_dataset("rajpurkar/squad", split="validation[:300]")
except Exception as e:
    raise RuntimeError(
        "Failed to download the SQuAD dataset. On Kaggle, make sure "
        "'Internet' is turned ON for this notebook (Settings panel)."
    ) from e

train_split = raw_train.train_test_split(test_size=0.2, seed=SEED)
train_dataset = train_split["train"]
val_dataset = train_split["test"]

print(f"Train set: {len(train_dataset)} | Validation set: {len(val_dataset)} | Test bench: {len(raw_eval)}")

# ---------------------------------------------------------
# 4. Model & tokenizer initialization
# ---------------------------------------------------------
MODEL_NAME = "valhalla/t5-small-qg-hl"
print(f"\n=== Initializing Tokenizer & Model ({MODEL_NAME}) ===")

try:
    # legacy=True matches the SentencePiece behavior this checkpoint was
    # originally trained/evaluated with. Switching to legacy=False changes
    # tokenization of certain whitespace/punctuation edge cases and can
    # silently hurt generation quality on older T5 checkpoints like this one.
    tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=True)
    baseline_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
except Exception as e:
    raise RuntimeError(
        f"Failed to download '{MODEL_NAME}' from the Hugging Face Hub. "
        "Check that Internet is enabled for this Kaggle notebook."
    ) from e

# The <hl> highlight token is already registered as an added token in this
# checkpoint's tokenizer files, so no manual add_special_tokens() call is
# needed. This assertion just makes that assumption explicit and fails
# loudly if a different base model is swapped in later.
assert "<hl>" in tokenizer.get_vocab() or "<hl>" in tokenizer.additional_special_tokens, (
    "Expected '<hl>' to be a known token for this checkpoint's tokenizer."
)

try:
    rouge = evaluate.load("rouge")
except Exception as e:
    raise RuntimeError(
        "Failed to download the 'rouge' metric definition. Check Internet access."
    ) from e

# ---------------------------------------------------------
# 5. Batched question generation
# ---------------------------------------------------------
def generate_questions_batch(model, tokenizer, contexts, answers, batch_size=32):
    """Batch size tuned for a T4's 16GB VRAM with t5-small."""
    model.eval()
    generated_questions = []

    prompts = [
        f"generate question: {highlight_answer_robust(ctx, ans)}"
        for ctx, ans in zip(contexts, answers)
    ]

    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i : i + batch_size]
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=64,
                num_beams=4,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        generated_questions.extend(decoded)

    return generated_questions


# ---------------------------------------------------------
# 6. Baseline evaluation
# ---------------------------------------------------------
print("\n=== Phase 1: Pretrained Baseline Evaluation ===")

test_contexts = [ex["context"] for ex in raw_eval]
test_answers = [ex["answers"]["text"][0] for ex in raw_eval]
test_references = [ex["question"] for ex in raw_eval]

print("Generating baseline predictions...")
baseline_preds = generate_questions_batch(baseline_model, tokenizer, test_contexts, test_answers)
baseline_scores = rouge.compute(predictions=baseline_preds, references=test_references)

del baseline_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ---------------------------------------------------------
# 7. Fine-tuning
# ---------------------------------------------------------
print("\n=== Phase 2: Model Fine-Tuning ===")


def preprocess_qg(examples):
    inputs = []
    targets = []

    for i in range(len(examples["context"])):
        ctx = examples["context"][i]
        ans = examples["answers"][i]["text"][0]
        q = examples["question"][i]

        highlighted = highlight_answer_robust(ctx, ans)
        inputs.append(f"generate question: {highlighted}")
        targets.append(q)

    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(targets, max_length=64, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_dataset.map(preprocess_qg, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess_qg, batched=True, remove_columns=val_dataset.column_names)

fine_tune_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Seq2SeqTrainer pads variable-length generated sequences across a batch
    # with -100 (the same sentinel used for label padding), not with
    # pad_token_id. Newer (Rust-backed) tokenizers raise OverflowError if
    # asked to decode a negative id, so -100 must be replaced here too,
    # exactly as is already done for `labels` below.
    preds = np.asarray(preds)
    preds = np.where(preds >= 0, preds, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    return {k: round(v, 4) for k, v in result.items()}


training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=20,
    weight_decay=0.01,
    # logging_steps is now derived from dataset size so the loss curve stays
    # readable regardless of how many training examples / epochs you use
    # (the previous fixed value of 50 was too coarse for a small run and
    # produced a "No log" epoch plus a stale repeated loss value).
    logging_steps=max(5, len(tokenized_train) // 16 // 4),
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=64,
    generation_num_beams=4,
    fp16=torch.cuda.is_available(),  # T4 has Tensor Cores -> fast native fp16
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=fine_tune_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=fine_tune_model, padding=True),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    # With more epochs on a small dataset, the model can start memorizing
    # instead of generalizing. This stops training automatically once
    # rouge1 (metric_for_best_model) hasn't improved for 3 consecutive
    # evaluations, and load_best_model_at_end=True (already set above)
    # ensures the best checkpoint - not the last one - gets saved.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

finetuned_model = T5ForConditionalGeneration.from_pretrained(SAVE_DIR).to(device)

# ---------------------------------------------------------
# 8. Post fine-tuning evaluation
# ---------------------------------------------------------
print("\n=== Phase 3: Fine-Tuned Model Evaluation ===")
finetuned_preds = generate_questions_batch(finetuned_model, tokenizer, test_contexts, test_answers)
finetuned_scores = rouge.compute(predictions=finetuned_preds, references=test_references)

# ---------------------------------------------------------
# 9. Comparative results
# ---------------------------------------------------------
print("\n=== Comparative Evaluation Summary ===")
comparison_df = pd.DataFrame(
    {
        "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L"],
        "Pretrained Baseline": [
            f"{baseline_scores['rouge1']:.4f}",
            f"{baseline_scores['rouge2']:.4f}",
            f"{baseline_scores['rougeL']:.4f}",
        ],
        "Fine-Tuned Model": [
            f"{finetuned_scores['rouge1']:.4f}",
            f"{finetuned_scores['rouge2']:.4f}",
            f"{finetuned_scores['rougeL']:.4f}",
        ],
    }
)
print(comparison_df.to_string(index=False))

# ---------------------------------------------------------
# 10. Breakdown: factoid vs. reasoning questions
# ---------------------------------------------------------
print("\n=== Question Type Performance Breakdown (Fine-Tuned Model) ===")
type_categories = defaultdict(list)

for i, ref_q in enumerate(test_references):
    first_word = ref_q.split()[0].lower() if ref_q.strip() else ""
    if first_word in ["who", "what", "where", "when"]:
        cat = "Factoid (Who/What/Where/When)"
    elif first_word in ["why", "how"]:
        cat = "Reasoning (Why/How)"
    else:
        cat = "Other Syntax"
    type_categories[cat].append((finetuned_preds[i], ref_q))

breakdown_data = []
for cat, pairs in type_categories.items():
    if not pairs:
        continue
    preds = [p for p, r in pairs]
    refs = [r for p, r in pairs]
    scores = rouge.compute(predictions=preds, references=refs)
    breakdown_data.append(
        {
            "Category": cat,
            "Count (n)": len(pairs),
            "ROUGE-1": f"{scores['rouge1']:.4f}",
            "ROUGE-2": f"{scores['rouge2']:.4f}",
            "ROUGE-L": f"{scores['rougeL']:.4f}",
        }
    )

breakdown_df = pd.DataFrame(breakdown_data)
print(breakdown_df.to_string(index=False))

# ---------------------------------------------------------
# 11. Qualitative error analysis
# ---------------------------------------------------------
print("\n=== Failure Mode & Qualitative Analysis ===")

rl_per_example = rouge.compute(
    predictions=finetuned_preds, references=test_references, use_aggregator=False
)["rougeL"]
# NOTE: this measures n-gram overlap between the generated question and the
# *source context* (not a standard ROUGE-against-reference score) - useful
# as a rough "is the model just copying the passage" signal, not a quality metric.
overlap_per_example = rouge.compute(
    predictions=finetuned_preds, references=test_contexts, use_aggregator=False
)["rouge1"]

error_analysis = []
for i in range(len(raw_eval)):
    error_analysis.append(
        {
            "rougeL": rl_per_example[i],
            "overlap": overlap_per_example[i],
            "pred": finetuned_preds[i],
            "ref": test_references[i],
            "context": test_contexts[i],
        }
    )

error_analysis.sort(key=lambda x: x["rougeL"])

print("Top 3 Worst Generations (Lowest ROUGE-L Alignment):")
for idx, item in enumerate(error_analysis[:3], 1):
    print(f"\n[{idx}] ROUGE-L Score: {item['rougeL']:.4f}")
    print(f"    Context:   {item['context'][:120]}...")
    print(f"    Reference: {item['ref']}")
    print(f"    Generated: {item['pred']}")

# ---------------------------------------------------------
# 12. Production demo
# ---------------------------------------------------------
print("\n=== Production Demo (Sample Inputs) ===")
demo_samples = [
    {"context": "Albert Einstein developed the theory of relativity in the early 20th century.", "answer": "Albert Einstein"},
    {"context": "Photosynthesis is the process used by plants to convert light energy into chemical energy.", "answer": "Photosynthesis"},
    {"context": "The Pyramids of Giza were constructed in Egypt as tombs for the Pharaohs.", "answer": "Egypt"},
]

demo_ctx = [d["context"] for d in demo_samples]
demo_ans = [d["answer"] for d in demo_samples]
demo_results = generate_questions_batch(finetuned_model, tokenizer, demo_ctx, demo_ans)

for sample, gen_q in zip(demo_samples, demo_results):
    print(f"Context:   {sample['context']}")
    print(f"Answer:    {sample['answer']}")
    print(f"Generated: {gen_q}")
    print("-" * 60)

[Setup] Installing helper packages (no force-reinstall, no version pins on core libs)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00

=== System Infrastructure ===
Device Target: cuda
GPU Hardware: Tesla T4
CUDA Version: 12.8
PyTorch Version: 2.10.0+cu128
Transformers Version: 5.0.0
Tokenizers Version: 0.22.2

=== Dataset Loading (SQuAD v1.1) ===


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



=== Phase 3: Fine-Tuned Model Evaluation ===

=== Comparative Evaluation Summary ===
 Metric Pretrained Baseline Fine-Tuned Model
ROUGE-1              0.4414           0.4379
ROUGE-2              0.2418           0.2327
ROUGE-L              0.4139           0.4054

=== Question Type Performance Breakdown (Fine-Tuned Model) ===
                     Category  Count (n) ROUGE-1 ROUGE-2 ROUGE-L
                 Other Syntax         52  0.3865  0.2028  0.3500
Factoid (Who/What/Where/When)        175  0.3970  0.1964  0.3607
          Reasoning (Why/How)         73  0.5760  0.3406  0.5552

=== Failure Mode & Qualitative Analysis ===
Top 3 Worst Generations (Lowest ROUGE-L Alignment):

[1] ROUGE-L Score: 0.0000
    Context:   Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015...
    Reference: What does AFC stand for?
    Generated: The Denver Broncos defeated the Carolina Panthers 24–10 to earn their third Super Bowl title